# Biopython으로 한타바이러스 서열 분석해보기
## 개요
제미나이랑 아 생물정보학은 뭐 해야되지? 하다가 팟! 떠올랐습니다. 보통 환자 데이터는 환자 개인의 것이기때문에... 내꺼 내가 쓰는 거 아니면 맘대로 쓸 수 없잖아요? 하지만 잘 찾아보면 쓸 수 있는 데이터가 있다 이겁니다. 쓰라고 올려둔! 니네들 갖다 쓰라고 해둔!! 그 데이터 중 하나가 바이러스의 유전정보입니다. 네. 

하고 많은 바이러스들 중 왜 하필 한타바이러스냐... 일단 이 녀석을 처음 발견한 사람이 한국인이고요... 한타바이러스가 딸린 식구가 진짜 많아요. 위키피디아 가서 보시면 얘네는 뭐 월드 와이드로 새끼치고 사나 싶으실겁니다. 바이러스한테 새끼친다는 표현은 좀 그렇지만 아무튼. 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import matplotlib.pyplot as plt

# Biopython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Align import AlignInfo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Seq import Seq

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import defaultdict

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일 
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요

# NCBI에서 데이터 가져오기

In [ ]:
# 바이러스 서열 다운로드
# 이게 근데 막 받으면 안되거든요? 한타바이러스 식구들은 둘쨰치고 저기가 데이터가 진짜 방대해요. 
virus_query = "Hantavirus[Organism] and complete genome"

print('Searching sequences... ') # 솔직히 이거 없으면 되는건지 불안하잖아요...

handle = Entrez.esearch(db="nucleotide", term=virus_query, retmax=300)
record = Entrez.read(handle)
id_list = record['IdList']

print(f"총 {len(id_list)}개의 표준 서열을 찾았습니다.") # 100개 최대로! 

In [ ]:
# 콤퓨타에 저-장
vir_sequence = []
for i, id in enumerate(id_list):
    print(f"Downloading sequence {i+1}/{len(id_list)}: {id}")
    handle = Entrez.efetch(db="nucleotide", id=id, rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    
    # record에 id와 seq가 다 들어가야되더라... (안되면 오류남 봤음)
    vir_sequence.append(record) 

# 파일로 저장
SeqIO.write(vir_sequence, "hantavirus_sequence.fasta", "fasta")
print('Done!')

In [ ]:
# 시퀀스 길이 체크 
for rec in vir_sequence:
    print(f"ID: {rec.id} | Length: {len(rec.seq)}")

In [ ]:
# 바이러스 DNA가 세그먼트별로 섞여있습니다. (S, M, L)
# 이거 분류 안하면 MSA 뻑나요. 
s_segments = [rec for rec in vir_sequence if 1600 <= len(rec.seq) <= 2000]
SeqIO.write(s_segments, "hantavirus_segmemt_s.fasta", "fasta")

# 세그먼트 ID를 이름으로 바꾸는 절차라고 보시면 됩니다. 
for rec in s_segments:
    rec.id = f"{rec.id}_{len(rec.seq)}"

print("Completed.")

# MSA 및 시각화

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try: 
    result = subprocess.run([muscle_exe, "-align", "hantavirus_segmemt_s.fasta", "-output", "Hantavirus_alignment_s.afa"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e: 
    print(f"MSA failed: {e}")

In [ ]:
# 다 됐으면 몇 개만 확인해보자. 
print("====== MSA Result ======")
alignment = AlignIO.read("Hantavirus_alignment_s.afa", "fasta") # FASTA 니네 확장자가 몇개냐... 

for record in alignment:
    print(f"{record.id[:10]:<15} : {record.seq[:50]}")

In [ ]:
# MSA 시각화
# 1. 염기 서열 직접 비교 
target_len = 1000
matrix = []
ids = []

# 첫 번째 서열을 기준으로 잡고 나머지와 비교
ref_seq = str(alignment[0].seq)[:target_len]

for record in alignment:
    # 기준 서열과 다르면 1, 같으면 0 (Gap '-' 제외 처리 유지)
    row = [1 if record.seq[i] != ref_seq[i] and record.seq[i] != '-' else 0 for i in range(target_len)]
    matrix.append(row)
    ids.append(record.id.split('.')[0])

# 2. 시각화 (imshow 대신 pcolormesh를 쓰면 더 선명합니다)
fig, ax = plt.subplots(figsize=(20, len(ids) * 0.3))
c = ax.pcolormesh(matrix, cmap='YlOrRd', edgecolors='none', antialiased=False)

# 레이블 및 그리드 설정
ax.set_yticks(np.arange(len(ids)) + 0.5)
ax.set_yticklabels(ids, fontsize=9)
ax.set_title(f"한타바이러스 S-Segment 염기 서열 변이 지도 (0-{target_len}bp)", fontsize=22, pad=30, fontweight='bold')
ax.set_xlabel("Nucleotide Position", fontsize=14, labelpad=15)
ax.set_ylabel("Sequence IDs", fontsize=14, labelpad=15)

# 100단위로 눈금 표시
ax.set_xticks(np.arange(0, target_len + 1, 100))
ax.grid(axis='x', color='black', linestyle=':', alpha=0.2)

# 세로 격자선을 추가하면 위치 파악이 더 쉽습니다.
ax.grid(axis='x', color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f"Hantavirus_S_DNA_Mutation_{target_len}bp.png", dpi=300)
plt.show()

# 섀넌 엔트로피 분석

## 변이가 안 되는 부분도 있나?

In [ ]:
# 코어 시퀀스는 어디? (바이러스라고 앞뒤 안가리고 다 변형하는거 아님)
def calculate_conservation(alignment):
    length = alignment.get_alignment_length()
    scores = []
    for i in range(length):
        column = alignment[:, i]
        most_common = max(column, key=column.count)
        score = column.count(most_common) / len(column)
        scores.append(score)
    return scores

scores = calculate_conservation(alignment)

print(f"해당 구간의 평균 보존율: {np.mean(scores)*100:.2f}%")

## 섀년 엔트로피 및 통계분석

In [ ]:
# 변이 핫스팟 찾기 (엔트로피 분석)
variation_scores = [1 - s for s in scores] # 위에서 계산한 그거 

# 핫스팟 시각화
plt.figure(figsize=(15, 5))
plt.plot(variation_scores, color='red', alpha=0.7)
plt.fill_between(range(len(variation_scores)), variation_scores, color='red', alpha=0.2)

plt.title("Viral Variation Hotspots (Potential Immune Evasion Sites)")
plt.xlabel("Sequence Position")
plt.ylabel("Variation Score (1-Shannon Entropy)")
plt.show()

### 섀넌 엔트로피에 대한 통계분석
- 귀무가설: Hantavirus S segment의 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다.
- 대립가설: Hantavirus S segment의 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

In [ ]:
# 여러분 이것도 통계분석이 됩니다. 
variation_scores = np.array(variation_scores)

mean_var = np.mean(variation_scores)
median_var = np.median(variation_scores)
iqr_var = np.percentile(variation_scores, 75) - np.percentile(variation_scores, 25)

print(f"Mean variation score: {mean_var:.4f}")
print(f"Median variation score: {median_var:.4f}")
print(f"IQR: {iqr_var:.4f}")

In [ ]:
# Define high-variation hotspots (top 10%)
threshold = np.percentile(variation_scores, 90)

hotspots = variation_scores[variation_scores >= threshold]
non_hotspots = variation_scores[variation_scores < threshold]

u_stat, p_value = mannwhitneyu(
    hotspots,
    non_hotspots,
    alternative="greater"
)

print(f"Hotspot threshold (90th percentile): {threshold:.4f}")
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.4e}") # 아 이거는 제가 소수점 조절을 못했어요... 하면 큰일나... 

- P-value < 0.001이므로 Hantavirus S segment의 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다는 귀무가설을 기각함. 
> Hantavirus S segment의 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

# Phylogenic tree

In [ ]:
# Phylogenic tree (full)

# 1. 거리 계산
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)

# 2. Phylogenic tree 생성(NJ)
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

# 이름이 모예요 
clean_name_dict = {}
for record in vir_sequence:
    # "Hantaan virus" 같은 종 이름만 추출
    species_name = record.description.split("segment")[0].split(",")[0].strip()
    # 너무 긴 학명 단어는 제거 (가독성 향상)
    species_name = species_name.replace("orthohantavirus", "").strip()
    clean_name_dict[record.id] = species_name

for leaf in tree.get_terminals():
    found = False
    for original_id, clean_name in clean_name_dict.items():
        if original_id.split('.')[0] in leaf.name.replace('_', '.'):
            leaf.name = str(clean_name) # 확실하게 문자열로 변환해서 주입
            found = True
            break
    if not found:
        leaf.name = leaf.name.split('_')[0]

# Inner 노드 소멸
for clade in tree.find_clades():
    if not clade.is_terminal():
        clade.name = None

# Phylogenic tree 시각화
fig = plt.figure(figsize=(26, 30), dpi=100)
axes = fig.add_subplot(1, 1, 1)
Phylo.draw(tree, axes=axes, do_show=False, show_confidence=False, label_func=lambda x: str(x.name) if x.is_terminal() else "")

# 오케이 트리 봐봐 
plt.rc('font', size=10) # 내부 글꼴 사이즈
plt.rc('axes', titlesize=20) # 제모옥은 이 크기로 하겠습니다 

plt.title("Hantavirus S-Segment Phylogenetic Tree") # 근데 이제 이걸 곁들인
plt.xlabel("Genetic Distance (Substitution per site)", fontsize=12)
plt.ylabel("Viral Strains", fontsize=12)
plt.tight_layout()
plt.savefig("Hantavirus_Final_Tree.png", dpi=300, bbox_inches='tight')
plt.show()

## 통계분석
- 귀무가설: 동일 clade 내 서열 유사도와 서로 다른 clade 간 서열 유사도에는 차이가 없다.
- 대립가설: 동일 clade 내 서열 유사도가 clade 간 서열 유사도보다 유의하게 높다.

In [ ]:
def pairwise_identity(seq1, seq2):
    matches = sum(a == b for a, b in zip(seq1, seq2) if a != '-' and b != '-')
    length = sum(a != '-' and b != '-' for a, b in zip(seq1, seq2))
    return matches / length if length > 0 else 0

def extract_clades(tree, cutoff=0.05):
    clade_map = {}
    clade_id = 0

    for clade in tree.find_clades():
        if clade.branch_length and clade.branch_length > cutoff:
            terminals = clade.get_terminals()
            for t in terminals:
                clade_map[t.name] = f"Clade_{clade_id}"
            clade_id += 1

    return clade_map

In [ ]:
clade_map = extract_clades(tree, cutoff=0.05)

# ID 정규화 (이거 중요)
normalized_clade_map = {}
for k, v in clade_map.items():
    normalized_clade_map[k.split('.')[0]] = v

In [ ]:
within_clade = []
between_clade = []

for rec1, rec2 in combinations(alignment, 2):
    id1 = rec1.id.split('.')[0]
    id2 = rec2.id.split('.')[0]

    if id1 not in normalized_clade_map or id2 not in normalized_clade_map:
        continue

    identity = pairwise_identity(str(rec1.seq), str(rec2.seq))

    if normalized_clade_map[id1] == normalized_clade_map[id2]:
        within_clade.append(identity)
    else:
        between_clade.append(identity)

print(len(within_clade), len(between_clade))

In [ ]:
u, p = mannwhitneyu(
    within_clade,
    between_clade,
    alternative="greater"
)

print(f'U-statistic: {u:.1f}')
print(f'p-value: {p:.4e}') # 그... 이게... 맞아요? 

- p < 0.001이므로 귀무가설을 기각한다. 
> 동일 clade 내 서열 유사도가 clade 간 서열 유사도보다 유의하게 높다.

### Bonus: 이펙트 사이즈

In [ ]:
effect_size = (
    np.median(within_clade) - np.median(between_clade)
)

print(f'effect_size: {effect_size:.4f}')

In [ ]:
n1 = len(within_clade)
n2 = len(between_clade)

rbc = 1 - (2 * u) / (n1 * n2)
print(f"Rank-biserial r: {rbc:.4f}")

- 쌍 비교의 99.6%에서 within-clade identity가 더 큼
- 이펙트 사이즈는 0.3479이다. 
> 변이는 특정 hotspot에 제한되어 있으며, 그 결과 유사한 변이 패턴을 공유하는 서열들이 phylogenetic clade로 강하게 묶였고, 이러한 clustering은 통계적으로도 극히 유의하였다.